# Speed benchmark: ridge map vs official OMP (same device, same dtype)

Settles the speed question fairly: every method runs on the **same device** (CPU, and GPU if available) with the **same data and dtype**, after a warm-up, with GPU synchronisation, median of several repeats.

* **Official OMP-64 / OMP-32**: `batch_omp` copied verbatim from mergekit 0.1.5, centred inputs, one call (earlier logs timed two calls together).
* **Ridge**: (a) fixed lambda (one solve), (b) fixed lambda on 5000 anchors, (c) lambda searched on held-out anchors as in the paper (5 validation solves + 1 final solve).
* Sweep over the number of new tokens K. Reads out the speed-up ratio OMP-64 / ridge per device.

Run all cells (Save Version > Run All). Send back `speed_benchmark.csv` and the printed tables.

In [ ]:
# ====== CONFIG (edit if needed) ======
import os, time, json, platform, subprocess, logging, statistics
from typing import Tuple
import numpy as np, torch
USE_REAL_EMBEDDINGS = True      # real GPT-2 / Pythia-160m / Qwen2.5-0.5B embedding matrices; falls back to synthetic data if download fails
K_SWEEP = [82, 300, 1000, 3000]  # number of new tokens to initialise
REPS = 5                         # timed repeats per setting (reduced automatically for slow runs)
LAMBDA_FIXED = 1e-2              # the fixed lambda used for "ridge, fixed lambda" (ablation: 1e-4..1e-2 change NLL by <= 0.06 nats)
RUN_SKLEARN_OMP = True           # also time our own scikit-learn OMP-32 (CPU only, K <= 300)
OUT_CSV = "speed_benchmark.csv"


In [ ]:
# ====== Official OMP, copied verbatim from mergekit 0.1.5 (mergekit/tokensurgeon/omp.py) ======
LOG = logging.getLogger('batch_omp')
def batch_omp(
    targets: torch.Tensor,
    candidate_points: torch.Tensor,
    k: int,
    eps: float = 1e-8,
    reorthogonalize_interval: int = 256,
) -> Tuple[torch.LongTensor, torch.Tensor]:
    """
    Batched Orthogonal Matching Pursuit (OMP) to select `k` points from `candidate_points` that best approximate each target in `targets`.

    Args:
        targets: (B, D) tensor of target vectors.
        candidate_points: (N, D) tensor of candidate points.
        k: Number of points to select (sparsity level).
        eps: Tolerance for numerical stability.
        reorthogonalize_interval: Number of iterations between reorthogonalization steps.

    Returns:
        selected_indices: (B, k) tensor of indices selected for each target.
        coeff: (B, k) tensor of coefficients for each selected point.
    """
    B, D = targets.shape
    N, _ = candidate_points.shape
    device = targets.device
    if k > N:
        raise ValueError(f"Cannot select {k} points from {N} candidates")
    work_dtype = (
        targets.dtype
        if targets.dtype in (torch.float32, torch.float64)
        else torch.float32
    )
    # Convert inputs to work_dtype
    targets_work = targets.to(dtype=work_dtype)
    points_work = candidate_points.to(dtype=work_dtype)
    # Preallocate tensors
    q = torch.zeros((B, D, k), dtype=work_dtype, device=device)
    r = torch.zeros((B, k, k), dtype=work_dtype, device=device)
    selected_indices = torch.zeros((B, k), dtype=torch.long, device=device)
    mask = torch.zeros((B, N), dtype=torch.bool, device=device)
    residuals = targets_work.clone()

    for t in range(k):
        rms_0 = residuals.norm(dim=1).mean()
        # Compute absolute inner products between residuals and points
        abs_inner = (residuals @ points_work.T).abs()  # (B, N)
        # Mask out already selected points
        abs_inner.masked_fill_(mask, -float("inf"))

        # Select new index with maximum correlation
        _, new_idx = torch.max(abs_inner, dim=1)  # (B,)
        selected_indices[:, t] = new_idx
        mask[torch.arange(B, device=device), new_idx] = True

        new_atom = points_work[new_idx]  # (B, D)
        if t == 0:
            r[:, 0, 0] = new_atom.norm(dim=1)
            norm = r[:, 0, 0].clamp(min=eps)
            q[:, :, 0] = new_atom / norm.unsqueeze(1)
        else:
            # Project onto existing basis
            projections = torch.bmm(
                q[:, :, :t].transpose(1, 2), new_atom.unsqueeze(-1)
            ).squeeze(-1)  # (B, t)
            residual = new_atom - torch.bmm(
                q[:, :, :t], projections.unsqueeze(-1)
            ).squeeze(-1)  # (B, D)
            norm = torch.clamp(torch.norm(residual, dim=1), min=eps)
            # Update R and Q
            r[:, :t, t] = projections
            r[:, t, t] = norm
            q[:, :, t] = residual / norm.unsqueeze(-1)

        if t > 0 and t % reorthogonalize_interval == 0:
            q_b = q[:, :, : t + 1]
            q_new, r_new = torch.linalg.qr(q_b, mode="reduced")
            r[:, : t + 1, : t + 1] = torch.bmm(r_new, r[:, : t + 1, : t + 1])
            q[:, :, : t + 1] = q_new

        qt_targets = torch.bmm(
            q[:, :, : t + 1].transpose(1, 2), targets_work.unsqueeze(-1)
        )  # (B, t+1, 1)
        approx = torch.bmm(q[:, :, : t + 1], qt_targets).squeeze(-1)
        residuals = targets_work - approx
        LOG.debug(f"OMP iteration {t}: RMS {rms_0} -> {residuals.norm(dim=1).mean()}")

    # Get final coefficients
    final_coeff = torch.linalg.solve_triangular(
        r[:, :k, :k],
        torch.bmm(q[:, :, :k].transpose(1, 2), targets_work.unsqueeze(-1)),
        upper=True,
    ).squeeze(-1)

    # Print residuals if we're yapping
    if LOG.isEnabledFor(logging.DEBUG):
        rt_approx = torch.bmm(
            final_coeff.unsqueeze(1), points_work[selected_indices]
        ).squeeze(1)
        residuals = targets_work - rt_approx
        LOG.debug(f"OMP final RMS: {residuals.norm(dim=1).mean()}")

    return selected_indices, final_coeff


# ====== Ridge map (ours), on any device ======
def unit_t(x): return x / x.norm(dim=-1, keepdim=True).clamp_min(1e-8)

def solve_ridge(X, Y, lam):
    A = X.T @ X
    A = A + (lam * torch.trace(A) / A.shape[0]) * torch.eye(A.shape[0], dtype=A.dtype, device=A.device)
    return torch.linalg.solve(A, X.T @ Y)

def ridge_fixed(Dc, Rc, Dterm, lam=LAMBDA_FIXED):
    """centre both spaces, scale donor side, ONE ridge solve with fixed lambda, direction of the new rows."""
    mu_d, mu_r = Dc.mean(0), Rc.mean(0)
    X, Y, T = Dc - mu_d, Rc - mu_r, Dterm - mu_d
    s = X.norm(dim=1).mean(); X, T = X / s, T / s
    return unit_t(T @ solve_ridge(X, Y, lam))

def ridge_search(Dc, Rc, Dterm, grid=(1e-4, 1e-3, 1e-2, 1e-1, 1.0), seed=0):
    """as in the paper: lambda chosen on held-out anchors by cosine (len(grid) validation solves + one final solve)."""
    mu_d, mu_r = Dc.mean(0), Rc.mean(0)
    X, Y, T = Dc - mu_d, Rc - mu_r, Dterm - mu_d
    s = X.norm(dim=1).mean(); X, T = X / s, T / s
    g = torch.Generator().manual_seed(seed); perm = torch.randperm(len(X), generator=g).to(X.device)
    nv = min(max(50, len(X) // 10), len(X) // 5); va, tr = perm[:nv], perm[nv:]
    best = None
    for lam in grid:
        cos = (unit_t(X[va] @ solve_ridge(X[tr], Y[tr], lam)) * unit_t(Y[va])).sum(1).mean().item()
        if best is None or cos > best[0]: best = (cos, lam)
    return unit_t(T @ solve_ridge(X, Y, best[1]))

def ridge_fixed_freq(Dc, Rc, Dterm, f_idx, lam=LAMBDA_FIXED):
    return ridge_fixed(Dc[f_idx], Rc[f_idx], Dterm, lam)

def official_omp(Dc, Rc, Dterm, k):
    """official batch_omp on centred inputs, then the new rows = coefficients applied to receiver anchor rows (one call)."""
    mu_d, mu_r = Dc.mean(0), Rc.mean(0)
    idx, co = batch_omp(Dterm - mu_d, Dc - mu_d, k)
    return torch.bmm(co.unsqueeze(1), (Rc - mu_r)[idx]).squeeze(1) + mu_r

# ====== Data: real embedding matrices (anchors = tokens with the same string in both vocabularies) ======
def load_pair(rec_name, don_name):
    from transformers import AutoTokenizer, AutoModel
    rt, dt = AutoTokenizer.from_pretrained(rec_name), AutoTokenizer.from_pretrained(don_name)
    R = AutoModel.from_pretrained(rec_name).get_input_embeddings().weight.detach().float().cpu()
    D = AutoModel.from_pretrained(don_name).get_input_embeddings().weight.detach().float().cpu()
    rv, dv = rt.get_vocab(), dt.get_vocab(); shared = sorted(set(rv) & set(dv))
    ri = torch.tensor([rv[t] for t in shared]); di = torch.tensor([dv[t] for t in shared])
    donor_only = torch.tensor(sorted(i for t, i in dv.items() if t not in rv))
    return D[di].contiguous(), R[ri].contiguous(), D[donor_only].contiguous()

def synthetic_pair(N, Dd, Dr, n_only, seed=0):
    g = torch.Generator().manual_seed(seed)
    Dm = torch.randn(N, Dd, generator=g); Rm = 0.05 * (Dm[:, :Dr] @ torch.randn(Dr, Dr, generator=g)) + 0.5 * torch.randn(N, Dr, generator=g)
    return Dm, Rm, torch.randn(n_only, Dd, generator=g)

PAIRS = {"GPT-2 <- Pythia-160m": ("gpt2", "EleutherAI/pythia-160m", (22781, 768, 768, 3500)),
         "GPT-2 <- Qwen2.5-0.5B": ("gpt2", "Qwen/Qwen2.5-0.5B", (28512, 896, 768, 3500))}
DATA = {}
for name, (rec, don, shp) in PAIRS.items():
    try:
        if not USE_REAL_EMBEDDINGS: raise RuntimeError("synthetic requested")
        DATA[name] = load_pair(rec, don); src = "real embeddings"
    except Exception as e:
        DATA[name] = synthetic_pair(*shp); src = f"SYNTHETIC data of the same size ({type(e).__name__})"
    print(f"{name}: anchors {tuple(DATA[name][0].shape)} -> {tuple(DATA[name][1].shape)}, donor-only tokens {len(DATA[name][2])}  [{src}]")

# ====== Environment ======
def cpu_name():
    try: return [l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")][0]
    except Exception: return platform.processor()
DEVICES = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
ENV = dict(torch=torch.__version__, cpu=cpu_name(), threads=torch.get_num_threads(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("ENV:", ENV)

# ====== Timing: same data, same device, same dtype; warm-up; synchronise; median of REPS ======
def sync(dev):
    if dev == "cuda": torch.cuda.synchronize()
def timeit(fn, dev, reps=REPS):
    ts = []
    if dev == "cuda":                      # warm-up (cuBLAS/cuSOLVER initialisation, allocator)
        fn(); sync(dev)
    for i in range(reps):
        sync(dev); t = time.perf_counter(); fn(); sync(dev); ts.append(time.perf_counter() - t)
        if ts[0] > 20 and i >= 0: break    # very slow settings: stop after the first repeat
    return statistics.median(ts), min(ts), len(ts)

rows = []
for pname, (Dall, Rall, Donly) in DATA.items():
    rng = np.random.default_rng(0)
    f_idx_full = torch.arange(min(5000, len(Dall)))       # stand-in for the 5000 most frequent anchors (cost depends on the number only)
    for K in K_SWEEP:
        sel = torch.as_tensor(rng.choice(len(Donly), size=K, replace=K > len(Donly)))
        Dterm_cpu = Donly[sel].contiguous()
        for dev in DEVICES:
            for dtype in (torch.float32, torch.float64):
                if dev == "cuda" and dtype == torch.float64 and K > 1000: continue
                Dc, Rc, Dt = Dall.to(dev, dtype), Rall.to(dev, dtype), Dterm_cpu.to(dev, dtype); fi = f_idx_full.to(dev)
                jobs = [("ridge, fixed lambda (all anchors)", lambda: ridge_fixed(Dc, Rc, Dt)),
                        ("ridge, fixed lambda (5000 anchors)", lambda: ridge_fixed_freq(Dc, Rc, Dt, fi)),
                        ("ridge, lambda searched (as in paper)", lambda: ridge_search(Dc, Rc, Dt))]
                if dtype == torch.float32 or K <= 1000:   # OMP in BOTH dtypes (batch_omp works in float32 and float64); float64 OMP at K=3000 on CPU is skipped (too slow)
                    jobs += [("OMP-64 official", lambda: official_omp(Dc, Rc, Dt, 64)), ("OMP-32 official", lambda: official_omp(Dc, Rc, Dt, 32))]
                for mname, fn in jobs:
                    med, mn, n = timeit(fn, dev)
                    rows.append(dict(pair=pname, K=K, device=dev, dtype=str(dtype).split(".")[-1], method=mname, median_s=med, min_s=mn, reps=n, ms_per_token=1000 * med / K))
                    print(f"{pname:22s} K={K:5d} {dev:4s} {str(dtype).split('.')[-1]:8s} {mname:38s} {med:8.3f} s  ({1000*med/K:7.2f} ms/token)", flush=True)
        if RUN_SKLEARN_OMP and K <= 300:
            from sklearn.linear_model import orthogonal_mp
            Dc, Rc, Dt = Dall.double().numpy(), Rall.double().numpy(), Dterm_cpu.double().numpy()
            un = lambda x: x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), 1e-8)
            t = time.perf_counter(); coef = orthogonal_mp(un(Dc).T, un(Dt).T, n_nonzero_coefs=32); med = time.perf_counter() - t
            rows.append(dict(pair=pname, K=K, device="cpu", dtype="float64", method="OMP-32 own (scikit-learn)", median_s=med, min_s=med, reps=1, ms_per_token=1000 * med / K))
            print(f"{pname:22s} K={K:5d} cpu  float64  OMP-32 own (scikit-learn)                 {med:8.3f} s", flush=True)

# ====== Report ======
import pandas as pd
df = pd.DataFrame(rows); df.to_csv(OUT_CSV, index=False)
print("\nENV:", ENV, "| LAMBDA_FIXED =", LAMBDA_FIXED, "| REPS =", REPS)
for dev in DEVICES:
  for dt in ("float32", "float64"):
    d = df[(df.device == dev) & (df.dtype == dt)]
    if d.empty: continue
    print(f"\n===== {dev.upper()} , {dt} : median seconds, ridge and OMP in the SAME dtype on the SAME device =====")
    pv = d.pivot_table(index=["pair", "K"], columns="method", values="median_s")
    if "OMP-64 official" in pv:
        pv["OMP-64 / ridge fixed"] = pv["OMP-64 official"] / pv["ridge, fixed lambda (all anchors)"]
        pv["OMP-64 / ridge searched"] = pv["OMP-64 official"] / pv["ridge, lambda searched (as in paper)"]
    print(pv.round(3).to_string())
print("\nReading guide: ratio > 1 means ridge is faster than official OMP-64 on that device; ratio < 1 means slower.")
